# Build Ground Truth Similarity Matrix

สร้าง sparse binary matrix ขนาด **36,807 × 36,807** จาก `merged_profiles.csv`
- **1** = profiles เป็นคนเดียวกัน (same `user_folder`)
- **0** = profiles เป็นคนละคน

ข้อมูล input: `merged_profiles.csv` รวม normalized text + user_folder + platform จาก 3 platforms (Twitter, Google+, Instagram)

ใช้ `scipy.sparse` เพื่อประหยัด memory (เก็บแค่ non-zero entries)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, save_npz
from itertools import combinations

## 1. Load Data

In [2]:
data_path = Path('/Users/tm/Documents/GitHub/Project-for-Work/data-for-project/merged_profiles.csv')

df_social = pd.read_csv(data_path).reset_index(drop=True)

print(f'merged_profiles: {df_social.shape}')
print(f'columns: {df_social.columns.tolist()}')
df_social[['userName', 'user_folder']].head()

merged_profiles: (36807, 22)
columns: ['userName', 'fullName', 'bio', 'location', 'externalUrl', 'pictureURL', 'bio_urls', 'bio_url_count', 'bio_mentions', 'bio_mentions_count', 'location_type', 'location_valid', 'latitude', 'longitude', 'url_count', 'externalUrl_clean', 'external_domain', 'user_folder', 'platform', 'source_folder', 'bigrams', 'outputProfileName']


,userName,user_folder
0,i3mawi,A3mawi
1,wolterskluwerespaa,A3Software
2,aalishanmatrix,aalishanmatrix
3,aaronbird,aaronbird
4,acfoto,AaronCohenArts


## 2. ตรวจสอบ user_folder

`merged_profiles.csv` มี `user_folder` อยู่แล้ว — ดึงมาจาก `combined_profiles.csv` และ merge กับ normalized data


In [3]:
N = len(df_social)

# user_folder มีอยู่แล้วใน merged_profiles
# profiles ที่ไม่มี label ให้ assign unique value (ไม่ match กับใคร)
null_mask = df_social['user_folder'].isna() | (df_social['user_folder'] == '')
df_social.loc[null_mask, 'user_folder'] = ['__no_label_' + str(i) for i in df_social.index[null_mask]]

print(f'Total profiles (N):              {N:,}')
print(f'Profiles with known user_folder: {(~null_mask).sum():,}')
print(f'Profiles without label:          {null_mask.sum():,}')
print(f'Platforms: {df_social["platform"].value_counts().to_dict()}')
df_social[['userName', 'user_folder', 'platform']].head(10)


Total profiles (N):              36,807
Profiles with known user_folder: 36,804
Profiles without label:          3
Platforms: {'twitter': 13960, 'googleplus': 11890, 'instagram': 10957}


,userName,user_folder,platform
0,i3mawi,A3mawi,googleplus
1,wolterskluwerespaa,A3Software,googleplus
2,aalishanmatrix,aalishanmatrix,googleplus
3,aaronbird,aaronbird,twitter
4,acfoto,AaronCohenArts,instagram
5,guantanamobae,Aaronthestrong,googleplus
6,aaronzlewis,aaronzlewis,twitter
7,iconicguy,abdelstewart,instagram
8,alexbindafernndez,abindafernandez,googleplus
9,abelserral,abser,googleplus


## 3. สร้าง group index: user_folder → list of row indices

In [4]:
groups = df_social.groupby('user_folder').groups  # dict: folder -> Index
match_groups = {k: list(v) for k, v in groups.items() if len(v) > 1}
total_pairs = sum(len(v) * (len(v) - 1) // 2 for v in match_groups.values())

print(f'Total user_folder groups:        {len(groups):,}')
print(f'Groups with >= 2 profiles:       {len(match_groups):,}')
print(f'Total positive pairs (i<j):      {total_pairs:,}')
print(f'Total entries (symmetric):       {total_pairs * 2:,}')

Total user_folder groups:        15,300
Groups with >= 2 profiles:       13,771
Total positive pairs (i<j):      29,247
Total entries (symmetric):       58,494


## 4. สร้าง Sparse Matrix (COO → CSR)

In [5]:
print(f'Building {N:,} x {N:,} sparse matrix...')

rows_list, cols_list = [], []
for indices in match_groups.values():
    for i, j in combinations(indices, 2):
        rows_list += [i, j]  # symmetric
        cols_list += [j, i]

rows_arr = np.array(rows_list, dtype=np.int32)
cols_arr = np.array(cols_list, dtype=np.int32)
data_arr = np.ones(len(rows_arr), dtype=np.int8)

matrix = coo_matrix((data_arr, (rows_arr, cols_arr)), shape=(N, N)).tocsr()

mem = (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 1024**2
print(f'Matrix shape:     {matrix.shape}')
print(f'Non-zero entries: {matrix.nnz:,}')
print(f'Density:          {matrix.nnz / (N * N):.8f}')
print(f'Memory (sparse):  ~{mem:.1f} MB')

Building 36,807 x 36,807 sparse matrix...
Matrix shape:     (36807, 36807)
Non-zero entries: 58,494
Density:          0.00004318
Memory (sparse):  ~0.4 MB


## 5. ตรวจสอบ Matrix

In [6]:
sample = list(match_groups.values())[0]
i, j = sample[0], sample[1]

print('=== Positive pair (same person) ===')
print(f'Profile {i}: {df_social.loc[i, "userName"]} | folder: {df_social.loc[i, "user_folder"]}')
print(f'Profile {j}: {df_social.loc[j, "userName"]} | folder: {df_social.loc[j, "user_folder"]}')
print(f'matrix[{i},{j}] = {matrix[i, j]}  <-ควรเป็น 1')
print(f'matrix[{j},{i}] = {matrix[j, i]}  <- ควรเป็น 1 (symmetric)')
print()

other = [k for k in list(match_groups.keys()) if k != df_social.loc[i, 'user_folder']][0]
k = match_groups[other][0]
print('=== Negative pair (different person) ===')
print(f'Profile {i}: folder: {df_social.loc[i, "user_folder"]}')
print(f'Profile {k}: folder: {df_social.loc[k, "user_folder"]}')
print(f'matrix[{i},{k}] = {matrix[i, k]}  <- ควรเป็น 0')

=== Positive pair (same person) ===
Profile 2770: caseycolettebyrddavis | folder: 1caseycolette
Profile 2771: 1caseycolette | folder: 1caseycolette
matrix[2770,2771] = 1  <-ควรเป็น 1
matrix[2771,2770] = 1  <- ควรเป็น 1 (symmetric)

=== Negative pair (different person) ===
Profile 2770: folder: 1caseycolette
Profile 2772: folder: 3twenty6
matrix[2770,2772] = 0  <- ควรเป็น 0


## 6. บันทึก Matrix

In [7]:
output_dir = Path('/Users/tm/Documents/GitHub/Project-for-Work/Train-Data')

matrix_path = output_dir / 'ground_truth_matrix.npz'
save_npz(matrix_path, matrix)
print(f'Saved: {matrix_path}  ({matrix_path.stat().st_size / 1024**2:.2f} MB)')

index_path = output_dir / 'ground_truth_matrix_index.csv'
df_social[['userName', 'user_folder']].to_csv(index_path, index=True, index_label='matrix_idx')
print(f'Saved: {index_path}')

Saved: /Users/tm/Documents/GitHub/Project-for-Work/Train-Data/ground_truth_matrix.npz  (0.11 MB)
Saved: /Users/tm/Documents/GitHub/Project-for-Work/Train-Data/ground_truth_matrix_index.csv


## 7. วิธีใช้ Matrix ใน Training

```python
from scipy.sparse import load_npz
import pandas as pd

# โหลด matrix และ index
matrix = load_npz('ground_truth_matrix.npz')           # shape (36807, 36807)
index_df = pd.read_csv('ground_truth_matrix_index.csv', index_col='matrix_idx')

# label สำหรับ pair (i, j)
label = int(matrix[i, j])      # 1 = same person, 0 = different

# positive profiles ทั้งหมดที่ match กับ profile i
positive_indices = matrix.getrow(i).nonzero()[1]

# ดึง all positive pairs
from scipy.sparse import find
rows_pos, cols_pos, _ = find(matrix)   # i < j pairs
mask = rows_pos < cols_pos
pairs = list(zip(rows_pos[mask], cols_pos[mask]))
```